# Menu ChatBot

In [1]:
from core.menu_parser import MenuParser
from scripts.exploratory_menu_analysis import *

# function to print section headers in the report
REPORT_WIDTH = 70
def print_section(title: str) -> None:
    print("\n" + "=" * REPORT_WIDTH)
    print(title)
    print("=" * REPORT_WIDTH)

This notebook walks through the implementation of the chatbot, and explores the data structure behind it, its limitations, and possible improvements.

## Menu Parsing and exploratory data analysis

The first step is to transform the raw menu JSON into a structured format that is easier to work with.

The `MenuParser` is designed as a consistent abstraction layer over the raw JSON data, organizing it into indexed structures that support efficient and reliable access.

A key design decision was to centralize normalization (for item names and sizes), ensuring that all lookups follow the same rules across the system. This avoids duplicating normalization logic in downstream components.

The parser builds multiple access patterns (by name, ID, and category), each serving a clear purpose. It also explicitly handles edge cases such as name collisions, making the system more robust.

In addition, the parser exposes a simple API (`get_item`, `get_price`) with a consistent response format, including structured error handling.

While the parser is intentionally simple, it already includes some query-related logic (e.g., price validation). In a larger system, this responsibility could be further separated, but for this scope it helps keep the overall design compact and practical.

In [2]:
parser = MenuParser('data/MenuDataTest.json')

# Lookup example
parser.get_item("nutty bowl")

# Price queries
parser.get_price("nutty bowl", "small")

{'success': True, 'data': {'price': 11.99, 'size': 'small'}, 'error': None}

## Exploratory Data Analysis


A quick exploratory analysis was performed to understand the structure and quality of the dataset.

In [3]:
run_exploratory_analysis()


1. DATA INVENTORY
Total raw item entries: 46
Total unique items by id: 46
Total indexed names: 46

Sample indexed item keys (first 10):
  1. dragon bowl
  2. superfood bowl
  3. warrior bowl
  4. green bowl
  5. nutty bowl
  6. tropical bowl
  7. kids bowl
  8. dessert bowl
  9. acai elixir
  10. go green

Name collisions: 0

2. PRICE EDGE CASES
Price range: $0.00 - $16.99
Zero/negative prices: 8
  They may need to be updated or removed from the menu.
  Items: ['temptation', 'power panini', 'superseed avocado toast', 'pb  chia jam toast', 'wholesome hummus toast', 'grilled cheese', 'kids pbj', 'kids sunsation smoothie']
Single-price items: 31
Multi-price items: 15
  Examples single: ['kids bowl', 'dessert bowl', 'acai elixir']
  Examples multi: ['dragon bowl', 'superfood bowl', 'warrior bowl']

3. NORMALIZATION AND LOOKUP CHECKS
Name normalization checks:
  OK: 'NUTTY  BOWL' -> NUTTY BOWL
  OK: 'GO GREEN!' -> GO GREEN
  OK: 'go green' -> GO GREEN
  OK: 'Go Green' -> GO GREEN
  OK: '  


The menu contains **46 unique items**, with consistent naming and no collisions after normalization, which enables reliable lookups.

Pricing is heterogeneous: most items have a single price, while others depend on size, requiring flexible handling of size-based queries. A few edge cases (e.g., zero-priced items) are present and, while acceptable for this exercise, would require careful validation in a production setting.

Nutrition data is sparse, with only a small subset of items providing this information, while discounts are more widely available and often linked to multiple items.

The menu is organized into a small number of categories, which supports grouping and filtering queries.

Overall, the dataset is well-structured, but its variability (in pricing, nutrition availability, and discount rules) directly influences how queries need to be handled.

## Chunking and RAG Pipeline



The system uses four specialized chunkers to transform structured data into retrieval-friendly text chunks:
- **ItemChunker**: Individual items with prices
- **CategoryChunker**: Category overviews
- **NutritionChunker**: Nutrition info (calories, dietary notes)
- **DiscountChunker**: Discount codes and eligibility

Each chunk is embedded and stored in ChromaDB for semantic retrieval.

In [4]:
from rag.chunkers.chunk_builder import ChunkBuilder

chunk_builder = ChunkBuilder(parser)
chunks, metadatas, ids = chunk_builder.build_all()

print(f"Total chunks created: {len(chunks)}")
print(f"\nChunk breakdown by type:")
from collections import Counter
types = Counter([m["type"] for m in metadatas])
for chunk_type, count in sorted(types.items()):
    print(f"  {chunk_type}: {count}")

print(f"\nExample item chunk:")
print(chunks[0])
print(f"\nExample discount chunk:")
discount_idx = next(i for i, m in enumerate(metadatas) if m["type"] == "discount")
print(chunks[discount_idx])

Total chunks created: 63

Chunk breakdown by type:
  category: 7
  discount: 7
  item: 46
  nutrition: 3

Example item chunk:
DRAGON BOWL is a menu item in the category: acai bowls.
            The item "DRAGON BOWL" has the following prices:
    - Medium: $14.49
- Large: $15.99

Example discount chunk:
Discount: 2 SM Bowls for $20

$20 off.


## Query Executor: Structured Lookups



For exact queries (prices, calories, categories), the system bypasses the LLM and uses deterministic lookups to ensure accuracy.


In [5]:
from core.query_executor import QueryExecutor

executor = QueryExecutor(parser)

# Price lookup
price_result = executor.execute({
    "intent": "item_price",
    "item_name": "nutty bowl",
    "size": "small",
    "category": None,
})
print(f"Price query result: {price_result}")

# Nutrition lookup
nutrition_result = executor.execute({
    "intent": "item_nutrition",
    "item_name": "go green",
    "size": None,
    "category": None,
})
print(f"Nutrition query result: {nutrition_result}")

# Category list
category_result = executor.execute({
    "intent": "category_list",
    "category": "salads",
    "item_name": None,
    "size": None,
})
print(f"Category query result:\n{category_result}")

Price query result: $11.99 (small)
Nutrition query result: GO GREEN is a smoothie with 240 calories.
Category query result:
Salads:
- supergreen goddess salad
- mighty med salad
- chimichurri steak  pot bowl
- green glow bowl
- power pesto chicken bowl


## Full Hybrid System



The RAGEngine orchestrates everything:
1. Intent detection (is this a price query? discount query? semantic query?)
2. Retrieval from vector store (using ChromaDB with optional type filtering)
3. QueryExecutor fallback (for structured intents)
4. LLM generation (for natural language synthesis)

Challenge questions demonstrate the full pipeline:

In [6]:
from rag.embeddings import EmbeddingModel
from rag.vector_store import VectorStore
from rag.llm_client import OllamaClient
from rag.rag_engine import RAGEngine

print("Building RAG system...")

# Embeddings and vector store
embedding_model = EmbeddingModel()
embeddings = embedding_model.embed_documents(chunks)

vector_store = VectorStore()
vector_store.add(chunks, embeddings, metadatas, ids)

# LLM and RAG engine
llm = OllamaClient()
rag = RAGEngine(embedding_model, vector_store, llm, executor)

print("RAG system ready.\n")

# Challenge questions
challenge_questions = [
    "What is the price of a small NUTTY BOWL?",
    "How many calories does the GO GREEN smoothie have?",
    "Which salads do you have?",
    "What discounts are available today?",
    "Which discounts include coupons?",
    "What items trigger a BOGO Any Smoothie discount?",
    "Is the price for Smoothie - ACAI ELIXIR the same in all channels?"
]

for i, question in enumerate(challenge_questions, 1):
    print(f"Q{i}: {question}")
    response = rag.run(question)
    print(f"A: {response}\n")

Building RAG system...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG system ready.

Q1: What is the price of a small NUTTY BOWL?
A: $11.99 (small)

Q2: How many calories does the GO GREEN smoothie have?
A: GO GREEN is a smoothie with 240 calories.

Q3: Which salads do you have?
A: We've got the following salads:
* Supergreen Goddess Salad
* MIGHTY MED SALAD
* Green Glow Bowl
* Power Pesto Chicken Bowl
* Chimichurri Steak & Pot Bowl
Hope that helps!

Q4: What discounts are available today?
A: Active discounts today:
- $1 OFF any Bowl or Smoothie ($1 off)
- $3.00 Off LG Bowl ($3 off)
- 2 SM Bowls for $20 ($20 off)
- BOGO 50% off Wrap or Salad (50% off)
- BOGO Any Smoothie (100% off)
- Military/Police Discount (20% off)
- Veteran Day Smoothie (100% off)

Q5: Which discounts include coupons?
A: Available coupon codes:
- 1000: $3.00 Off LG Bowl
- Military: Military/Police Discount

Q6: What items trigger a BOGO Any Smoothie discount?
A: The BOGO Any Smoothie discount is triggered when you buy one smoothie.

Q7: Is the price for Smoothie - ACAI ELIXIR the

## More Query Examples

Try different kinds of queries to see how the system handles them:

In [7]:
# Additional query examples
more_queries = [
    "What's the price of a large smoothie?",
    "Tell me about the acai elixir",
    "What items have protein in them?",
    "What bowls do you have?",
    "Are there any vegetarian options?",
    "What's the cheapest item on the menu?",
    "Do you have any cold beverages?",
    "What's in the power pesto bowl?",
]

print("Additional query examples:\n")
for question in more_queries:
    print(f"Q: {question}")
    response = rag.run(question)
    print(f"A: {response}\n")

Additional query examples:

Q: What's the price of a large smoothie?
A: Please specify the smoothie name with size 'large'. Examples: ACAI ELIXIR, GO GREEN, MATCHA MADNESS, TROPICAL PARADISE, DRAGON SMOOTHIE.

Q: Tell me about the acai elixir
A: The ACIA ELIXIR is a smoothie, and it's priced at $8.49 by default! That's all I can tell you from the menu information. If you'd like to know more or have any questions, feel free to ask!

Q: What items have protein in them?
A: Based on the menu information, the items with protein in them are:
* CHIMICHURRI STEAK WRAP (sandwiches)
* POWER PANINI (sandwiches)
* GRILLED CHEESE (sandwiches)
* TROPICAL PARADISE (smoothies)

Q: What bowls do you have?
A: We have WARRIOR BOWL, GREEN BOWL, KIDS BOWL, DESSERT BOWL, DRAGON BOWL, TROPICAL BOWL, and NUTTY BOWL!

Q: Are there any vegetarian options?
A: Yes, there are several vegetarian options on the menu! SUPERSEED AVOCADO TOAST and WHOLESOME HUMMUS TOAST are both sandwiches that don't contain meat. Addi

## Consistency Check: Same Query x5

This test runs the same question 5 times and compares outputs to measure response stability.

In [8]:
from collections import Counter

def run_consistency_test(rag, queries, runs=5):
    if isinstance(queries, str):
        queries = [queries]

    all_counts = {}

    for test_query in queries:
        results = []

        print("=" * 70)
        print(f"Running the same query {runs} times...")
        print(f"Query: {test_query}\n")

        for i in range(1, runs + 1):
            answer = rag.run(test_query)
            results.append(answer)
            print(f"Run {i}: {answer}\n")

        counts = Counter(results)
        all_counts[test_query] = counts

        print("=" * 70)
        print("Comparison summary")
        print("=" * 70)
        print(f"Unique outputs: {len(counts)} of {runs}")

    return all_counts

# Test 2 or 3 queries here
test_queries = [
    "Tell me about the acai elixir",
    "Are there any vegetarian options?",
    "What's in the power pesto bowl?",
]

consistency_summary = run_consistency_test(rag, test_queries, runs=5)

Running the same query 5 times...
Query: Tell me about the acai elixir

Run 1: The AC AI ELIXIR is a smoothie! It's got a default price of $8.49. That's all I've got on this one, but it sounds like a tasty treat!

Run 2: The ACAI ELIXIR is a smoothie! It's part of our smoothies category, and its default price is $8.49. Enjoy!

Run 3: The ACAI ELIXIR is a smoothie menu item priced at $8.49 by default! That's all I have on that one for now - if you'd like to know more, feel free to ask!

Run 4: The Acai Elixir! It's a smoothie, and it costs $8.49 by default. That's all I have on this menu item! Would you like to know more about our other drinks or bowls?

Run 5: The ACAI ELIXIR! It's a smoothie, and it costs $8.49 by default. Enjoy!

Comparison summary
Unique outputs: 5 of 5
Running the same query 5 times...
Query: Are there any vegetarian options?

Run 1: Yes, there are vegetarian options! SUPERSEED AVOCADO TOAST and WHOLESOME HUMMUS TOAST are both sandwiches that fit the bill. Addition

## Fuzzy Matching and Robust Input Handling


In real-world usage, user input is often imperfect: item names may contain typos, extra characters, or slight variations.

To handle this, the system includes a lightweight fuzzy matching step in the executor layer. Before performing any lookup, item names are normalized and matched against known menu entries, allowing the system to recover from common misspellings.

The examples below demonstrate how the system maps noisy inputs (e.g., "nuty bawl", "go greeen") to the correct menu items, and still returns accurate results.

In [9]:
# Final: fuzzy input handling in the bot (end-to-end via rag.run)
from importlib import reload
import rag.rag_engine as rag_engine_module

if not all(name in globals() for name in ["embedding_model", "vector_store", "llm", "executor"]):
    raise RuntimeError("Run the Full Hybrid System setup cell first.")

# Reload RAG engine to pick up latest fuzzy-input changes
reload(rag_engine_module)
RAGEngine = rag_engine_module.RAGEngine
rag = RAGEngine(embedding_model, vector_store, llm, executor)

fuzzy_bot_queries = [
    "Tell me about the acai elixer",
    "What is the price of a small nuty bawl?",
    "How many calories does go greeen smoothie have?",
]

print("Fuzzy bot input examples:\n")
for q in fuzzy_bot_queries:
    print(f"Q: {q}")
    print(f"A: {rag.run(q)}\n")

Fuzzy bot input examples:

Q: Tell me about the acai elixer
A: The ACIA ELIXIR! It's a delicious smoothie in our smoothies category, and it normally costs $8.49. Would you like to know more or try one out?

Q: What is the price of a small nuty bawl?
A: $11.99 (small)

Q: How many calories does go greeen smoothie have?
A: GO GREEN is a smoothie with 240 calories.



## Overall Performance

The system performs well on structured queries such as prices, calories, and category listings. The deterministic executor ensures accurate and consistent results for these cases, reducing reliance on the LLM for critical values.

For more open-ended queries, the RAG pipeline produces generally correct and natural responses when the relevant information is explicitly present in the dataset.

However, several limitations remain.

**Response consistency is limited**: the same query can produce different outputs across runs, reflecting the non-deterministic nature of the LLM. 

**System depends heavily on the underlying model**: Issues such as inconsistent phrasing, minor name distortions, and weak reasoning in some queries are largely tied to model limitations rather than retrieval.

**Queries that require aggregation or global reasoning (e.g., identifying the cheapest item) are not handled reliably**, as the system does not explicitly compute over the dataset and may be affected by noisy values like $0.00 prices. This is might be related to the usage of a lightweight LLM.

Some queries rely on implicit assumptions not present in the data (e.g., vegetarian options or discount conditions), which can lead to partially incorrect or inconsistent answers. 

Finally, when information is missing (e.g., ingredients), the system avoids hallucination but returns limited responses.This reflects a deliberate trade-off favoring data fidelity over completeness.

### Possible Improvements

- Use a stronger or more deterministic LLM to improve consistency
- Add simple aggregation logic for queries requiring comparisons
- Improve intent detection and routing
- Enrich the dataset (e.g., nutrition, ingredients, discount rules)
- Apply stricter output constraints to reduce variability